In [1]:
import torch
import numpy as np


class WeaklyConvexProblem:
    """
    Defines the objective function phi(x) = f(x) + r(x).
    Based on Example 2.1 (Phase Retrieval) or simple |x^2 - 1|.
    f_objective: The stochastic loss function f(x)
    """

    def __init__(self, f_objective, r_objective, model_gen, rho = 2.0):
        self.f = f_objective
        self.r = r_objective
        self.get_model = model_gen
        self.rho = rho # Used to guide the Solver's beta_t

    def objective(self, x, data_batch=None):
        return self.f(x, data_batch) + self.r(x)

In [25]:
class ModelBasedSolver:
    """
    Implements Algorithm 1.2: Stochastic Model-Based Minimization.
    """
    def __init__(self, problem, data, x_init, T, batch_size=1, gamma=0.1):
        self.prob = problem
        self.data = data # The population P
        self.x = x_init.clone().detach()
        self.T = T
        self.gamma = gamma # Step-size parameter
        self.batch_size = batch_size
        self.history = []
        
    def get_beta(self, t):
        # The paper requires beta_t > rho 
        # We add a small constant (e.g., 0.1) to ensure strict strong convexity
        rho_hat = self.prob.rho + 0.1 
        return rho_hat + (1.0 / self.gamma) * np.sqrt(t + 1.0)

    def run(self):
        
        for t in range(self.T):            
            beta_t = self.get_beta(t)
            
            indices = torch.randint(0, len(self.data), (self.batch_size,))
            batch = self.data[indices]
            
            x_t = self.x.clone().detach()
            g_xt = self.prob.get_model(x_t, batch)
            
            # SUBPROBLEM: x_{t+1} = argmin { f_xt(y) + (1/2*alpha) * ||y-x_t||^2 }
            y = x_t.clone().requires_grad_(True)
            inner_opt = torch.optim.LBFGS([y], lr=1, max_iter=20)

            def closure():                
                inner_opt.zero_grad()
                model_val = g_xt(y) + self.prob.r(y) # Add the regularizer r(y)
                penalty = (beta_t / 2) * torch.norm(y - x_t)**2
                loss = model_val + penalty
                loss.backward()
                return loss

            inner_opt.step(closure)
            self.x = y.detach().clone()
            
            if t % 50 == 0:
                # Calculate the True Objective (over the whole population)
                # This represents the 'Population Risk'
                with torch.no_grad():
                    mean_f = torch.mean(self.prob.f(self.x, self.data))
                    reg_val = self.prob.r(self.x)
                    total_phi = mean_f + reg_val
                    
                print(f"Iter {t:4}: x = {self.x.item():.4f} | True phi(x) = {total_phi.item():.4f}")

        return self.x

In [26]:
def max_parabola_phi(x, batch):
    """The 'True' Objective: max(x^2, (x-4)^2)"""
    f1 = (x - batch[:, 0])**2
    f2 = (x - batch[:, 1])**2
    return torch.max(f1, f2)


def max_parabola_model_gen(x_t, batch):
    """
    Returns the Stochastic Model for the Pointwise Max.
    Based on Example 2.6 and Section 4.2[cite: 984].
    """
    # Detach constants for the inner loop
    x_t_val = x_t.detach()
    f1_vals = (x_t_val - batch[:, 0])**2
    f2_vals = (x_t_val - batch[:, 1])**2

    g1_vals = 2 * (x_t_val - batch[:, 0])
    g2_vals = 2 * (x_t_val - batch[:, 1])

    def model(y):
        # Linearize components for each sample in batch, then average
        l1 = f1_vals + g1_vals * (y - x_t_val)
        l2 = f2_vals + g2_vals * (y - x_t_val)
        return torch.mean(torch.max(l1, l2))
    
    return model

def elastic_net_regularizer(x, l1_ratio=0.5, mu=0.01):
    """
    Combines L1 and L2 regularization. 
    Matches the 'regularized population risk' framework[cite: 13, 20].
    """
    l1_term = l1_ratio * torch.norm(x, p=1)
    l2_term = (1 - l1_ratio) * 0.5 * torch.norm(x, p=2)**2
    # Ensure the result is a tensor attached to the same device as x
    return mu * (l1_term + l2_term)

In [11]:
# Create a population of 1000 data points (targets for the parabolas)
# Center them around 0 and 4 so the "average" intersection is near 2
torch.manual_seed(42)
population_data = torch.randn(1000, 2)
population_data[:, 0] += 0.0  # Centers for f1
population_data[:, 1] += 4.0  # Centers for f2

In [27]:
# 1. Setup Problem
# rho=2 because the components (x-xi)^2 have a second derivative of 2
prob = WeaklyConvexProblem(
    f_objective=max_parabola_phi, 
    r_objective=elastic_net_regularizer, 
    model_gen=max_parabola_model_gen,
    rho=2.0
)

# 2. Run with Mini-Batching
solver = ModelBasedSolver(
    problem=prob, 
    data=population_data, 
    x_init=torch.tensor([10.0]), 
    T=1000, 
    batch_size=32, 
    gamma=0.5
)

final_x = solver.run()
print(f"Final sampled x: {final_x.item():.4f}")

Iter    0: x = 5.1482 | True phi(x) = 27.7060
Iter   50: x = 2.0375 | True phi(x) = 7.4016
Iter  100: x = 1.9812 | True phi(x) = 7.3995
Iter  150: x = 1.9949 | True phi(x) = 7.3979
Iter  200: x = 1.9966 | True phi(x) = 7.3978
Iter  250: x = 2.0373 | True phi(x) = 7.4016
Iter  300: x = 2.0113 | True phi(x) = 7.3982
Iter  350: x = 2.0143 | True phi(x) = 7.3984
Iter  400: x = 1.9906 | True phi(x) = 7.3983
Iter  450: x = 2.0175 | True phi(x) = 7.3987
Iter  500: x = 2.0700 | True phi(x) = 7.4121
Iter  550: x = 1.9230 | True phi(x) = 7.4193
Iter  600: x = 1.9815 | True phi(x) = 7.3994
Iter  650: x = 1.9799 | True phi(x) = 7.3997
Iter  700: x = 2.0415 | True phi(x) = 7.4026
Iter  750: x = 1.9238 | True phi(x) = 7.4190
Iter  800: x = 1.9453 | True phi(x) = 7.4095
Iter  850: x = 2.0236 | True phi(x) = 7.3994
Iter  900: x = 1.9901 | True phi(x) = 7.3983
Iter  950: x = 1.9911 | True phi(x) = 7.3982
Final sampled x: 1.9896
